In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, hour, dayofweek, month
from pyspark.sql.types import *
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import joblib
import os

In [ ]:
spark = SparkSession.builder \
    .appName("FlightDelayTraining") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark ready!")

#  load  data 
BASE_PATH = "/home/ahmed-refat/Desktop/flights & airports"

flights_df = spark.read.csv(
    f"{BASE_PATH}/flights.csv",
    header=True,
    inferSchema=True
)


weather_df = spark.read.csv(
    f"{BASE_PATH}/weather.csv",
    header=True,
    inferSchema=True
)

airports_df = spark.read.csv(
    f"{BASE_PATH}/airports.csv",
    header=True,
    inferSchema=True
)

print(f"Flights: {flights_df.count()} rows")
print(f"Weather: {weather_df.count()} rows")
print(f"Airports: {airports_df.count()} rows")


# هبص علي الداتا 
print("=== FLIGHTS ===")
flights_df.printSchema()
flights_df.show(3)

print("=== WEATHER ===")
weather_df.printSchema()
weather_df.show(3)

print("=== AIRPORTS ===")
airports_df.printSchema()
airports_df.show(3)

#-----------------------------------
# clean filght dataset
flights_clean = flights_df.select(
    "Month", "DayofMonth", "DayOfWeek",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "DepDelay", "Distance",
    "Cancelled", "Diverted",
    "WeatherDelay", "CarrierDelay"
) \
.filter(col("Cancelled") == 0) \
.filter(col("Diverted") == 0) \
.filter(col("DepDelay").isNotNull()) \
.filter(col("Origin").isNotNull()) \
.withColumn("FlightDate", col("FlightDate").cast("string"))

print(f"Flights after cleaning: {flights_clean.count()} rows")

#-------------------------------------
# clean weather 
weather_clean = weather_df.select(
    to_date(col("datetime")).alias("date"),
    col("temp").alias("temperature"),
    col("windspeed").alias("wind_speed"),
    col("windgust").alias("wind_gust"),
    col("precip").alias("precipitation"),
    col("visibility"),
    col("humidity"),
    col("cloudcover"),
    col("severerisk")
) \
.filter(col("temperature").isNotNull()) \
.filter(col("wind_speed").isNotNull())

print(f"Weather after cleaning: {weather_clean.count()} rows")

#------------------------------------------
# clean airport
airports_clean = airports_df.select(
    col("iata_code"),
    col("type").alias("airport_type"),
    col("latitude_deg").alias("airport_lat"),
    col("longitude_deg").alias("airport_lon"),
    col("elevation_ft")
) \
.filter(col("iata_code").isNotNull())

print(f"Airports after cleaning: {airports_clean.count()} rows")



#-------------------------------------------------------------
#join 

# flights + airports
df = flights_clean.join(
    airports_clean,
    flights_clean.Origin == airports_clean.iata_code,
    "left"
).drop("iata_code")

# flights + weather
df = df.join(
    weather_clean,
    to_date(col("FlightDate")) == weather_clean.date,
    "left"
).drop("date")

print(f"After join: {df.count()} rows")
df.show(3)


#---------------------------------------------------------------
#  اضافه عمود جديد و تحديد الفيتشرز لترين المودل 

from pyspark.sql.functions import StringIndexer
from pyspark.ml.feature import StringIndexer as MLStringIndexer

# عمل target column
df = df.withColumn(
    "is_delayed",
    when(col("DepDelay") > 15, 1).otherwise(0)
)

# اختيار الـ features
final_df = df.select(
    # flight features
    col("Month").cast("int"),
    col("DayofMonth").cast("int"),
    col("DayOfWeek").cast("int"),
    col("CRSDepTime").cast("int"),
    col("Distance").cast("float"),
    col("Reporting_Airline"),
    # airport features
    col("airport_lat").cast("float"),
    col("airport_lon").cast("float"),
    col("elevation_ft").cast("float"),
    # weather features
    col("temperature").cast("float"),
    col("wind_speed").cast("float"),
    col("wind_gust").cast("float"),
    col("precipitation").cast("float"),
    col("visibility").cast("float"),
    col("humidity").cast("float"),
    col("cloudcover").cast("float"),
    col("severerisk").cast("float"),
    # target
    col("is_delayed")
) \
.filter(col("is_delayed").isNotNull()) \
.dropna()

print(f"Final dataset: {final_df.count()} rows")
final_df.show(3)

In [ ]:
#---------------------------------------------------
# تدريب المودل 

# تحويل لـ pandas
pdf = final_df.toPandas()

# encode الـ airline
pdf["Reporting_Airline"] = pdf["Reporting_Airline"].astype("category").cat.codes

# تقسيم الداتا
X = pdf.drop("is_delayed", axis=1)
y = pdf["is_delayed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Delay ratio: {y.mean():.2%}")


#-----------------------------------------------------------
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y[y==0]) / len(y[y==1]),  # لمعالجة imbalanced data
    random_state=42,
    eval_metric="logloss"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10
)

print("Training done!")

In [ ]:
#تقيم المودل 
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["On-Time", "Delayed"]))

In [ ]:
# حفظ المودل 
MODEL_PATH = f"{BASE_PATH}/model"
os.makedirs(MODEL_PATH, exist_ok=True)

# حفظ المودل
joblib.dump(model, f"{MODEL_PATH}/xgboost_delay_model.pkl")

# حفظ الـ feature names عشان نستخدمهم في الـ stream
feature_names = list(X.columns)
joblib.dump(feature_names, f"{MODEL_PATH}/feature_names.pkl")

print(f"Model saved at: {MODEL_PATH}/xgboost_delay_model.pkl")
print(f"Features: {feature_names}")